# Chapter 8 — Blazing Fast Lookup with Hash Tables

## Why this chapter matters

Chapter 3 established `O(1) < O(log N) < O(N)`. This chapter introduces a 
data structure — the hash table — that gets you `O(1)` lookups even when 
searching for a specific piece of data, something arrays can only do `O(N)` 
(unordered) or `O(log N)` (ordered, binary search) for.

Python's `dict` **is** a hash 
table. This chapter explains the mechanism underneath it — the same 
mechanism as in the "what is hashing" 
conversation during Chapter 3 of Dictionaries and Sets.

## What a hash table actually is

A list of paired values: a **key** and a **value**, associated with each 
other. Different languages call this different things — hash, map, hash map, 
dictionary, associative array — all the same underlying structure.

**Why lookup is O(1)**: the key itself is run through a **hash function**, 
which converts it into a number. That number is used directly as an index 
into an underlying array-like storage. No searching happens — the key tells 
you exactly where to look, in one step.

In [1]:
menu = {"french fries": 0.75, "hamburger": 2.5, "hot dog": 1.5, "soda": 0.6}

menu["french fries"]   # O(1) — hash the key, jump directly to its cell

0.75

## The one requirement for a valid hash function

A hash function must be **deterministic**: the same input must always 
produce the same output. A function using randomness or the current time 
would be invalid — it would place a value one place on write, then look in 
the wrong place on read.

The book uses a simplified toy hash function throughout this chapter: 
convert each letter to a number (A=1, B=2, ...) and multiply them together. 
`BAD` → `2 × 1 × 4 = 8`. Real hash functions are far more complex, but this 
keeps the mechanics visible.

## Insertion and lookup, mechanically

**Insertion**: hash the key → get a number → store the value at that 
"cell" (index).

**Lookup**: hash the *same* key again → get the *same* number → go directly 
to that cell → return whatever's stored there.

Because both insertion and lookup use the identical hashing process, the key 
always leads you back to exactly where its value was placed.

## One-directional lookups — an important limitation

Fast `O(1)` lookup **only** works key → value. Going the other direction 
(finding a key given a value) requires scanning every entry — `O(N)` — since 
values were never hashed or placed by their own content.

This also explains two rules you already know from Python dicts:
- **Keys must be unique** — writing to an existing key overwrites the old 
  value, since the key always hashes to the same cell.
- **Values can repeat freely** — nothing about the hashing process cares 
  about value uniqueness.

## Collisions — when two keys hash to the same cell

A **collision** happens when two different keys hash to the same number. 
Example with the toy hash function: `DAB = 4×1×2 = 8`, same as `BAD = 2×1×4 = 8`.

**Separate chaining** is the classic fix: instead of storing a single value 
in a cell, store a small **array** of key-value pairs. On lookup, hash the 
key to find the right cell, then do a short **linear search** within that 
cell's array to find the exact matching key.

**Consequence**: if collisions are bad enough, lookup degrades from `O(1)` 
toward `O(N)` — in the worst case (everything hashed into one cell), a hash 
table performs no better than a plain array. This is why hash table quality 
depends entirely on minimizing collisions.

## What makes a hash table efficient — three factors

1. **How much data is stored** relative to available cells
2. **How many cells exist**
3. **The quality of the hash function** — a good one spreads keys evenly 
   across all available cells; a poor one (e.g. one that only ever outputs 
   values 1–9) wastes most of the table and causes constant collisions

**The load factor** — the ratio of stored elements to available cells. Rule 
of thumb: **7 elements per 10 cells** (load factor ≈ 0.7) balances two 
competing goals:
- Too few cells → frequent collisions, degraded performance
- Too many cells → wasted memory

When a hash table grows past its ideal load factor, the language runtime 
automatically **resizes** it (more cells, keys re-hashed and redistributed) — 
this is handled for you; you don't manage it manually in Python.

## Hash tables for organization vs. hash tables for speed — two distinct use cases

**Organization**: when data is *naturally* paired — a menu (item → price), 
a thesaurus (word → synonym), vote tallies (candidate → count). This is the 
"normal" way you already use dicts.

**Speed** (the more surprising use case): using a hash table purely as a 
fast **lookup index**, even when the data isn't naturally paired at all. You 
store items as *keys*, with a throwaway value (like `True`), purely to get 
`O(1)` "have I seen this before?" checks.

In [ ]:
# the array subset problem, demonstrating this exact technique: 

def is_subset(array1, array2):
    # Determine which is larger, for efficiency
    if len(array1) > len(array2):
        larger_array, smaller_array = array1, array2
    else:
        larger_array, smaller_array = array2, array1

    # Build a hash table "index" from the larger array — O(N) once
    hash_table = {}
    for value in larger_array:
        hash_table[value] = True

    # Check every item in the smaller array in O(1) each — no nested loop
    for value in smaller_array:
        if value not in hash_table:
            return False

    return True

is_subset(["a","b","c","d","e","f"], ["b","d","f"])   # True
is_subset(["a","b","c","d","e","f"], ["b","d","f","h"])   # False

False

**Why this matters**: the naive nested-loop version (loop through the 
smaller array, and for each item, loop through the larger array checking 
for a match) is `O(N × M)` — for every item in the smaller array, you 
potentially scan the *entire* larger array.

Using a hash table as an index instead: build it once (`O(N)`, one pass over 
the larger array), then check membership for every item in the smaller array 
at `O(1)` each. Total: `O(N)` — a **huge** improvement over `O(N × M)`, and 
directly the same "avoid checking `in` against a list" idea you already 
internalized back in your dedup exercise in Week 1 (`num not in list` being 
O(n) vs. a set/dict lookup being O(1)).

**The pattern to remember**: whenever an algorithm needs to search the same 
collection repeatedly, building a hash table index once up front turns N 
separate O(N) searches into N separate O(1) lookups.

# EXERCISES

## all four require O(N), so a hash-table-as-index approach is the intended tool for each

1. **Array intersection**: given two arrays, return a new array of values 
   present in *both*. Must be `O(N)` — don't use a built-in intersection method.
2. **First duplicate value**: given an array of strings, return the first 
   value that appears twice.
3. **Missing letter**: given a string containing every letter of the 
   alphabet except one, find the missing letter.
4. **First non-repeated character**: given a string, return the first 
   character that appears exactly once (e.g. `"minimum"` → `"n"`, since `n` 
   and `u` both appear once, but `n` comes first).

In [ ]:
# 1)
array1 = [1, 2, 3, 4, 5]
array2 = [0, 2, 4, 6, 8]


